# Road Damage Detection: run on Colab / Kaggle GPU
Runtime > Change runtime type > GPU. Run cells top to bottom and send the outputs of the cells marked **SEND** back for review.

**Before using RDD2022 check its license on the Figshare page** (the downloader prints what Figshare reports).

In [ ]:
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available())

## 1. Get the repo
Upload `road-damage-detection-platform.zip` to Google Drive (MyDrive), then:

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp /content/drive/MyDrive/road-damage-detection-platform.zip /content/
!unzip -q -o /content/road-damage-detection-platform.zip -d /content/rdd
%cd /content/rdd
!pip install -q torchmetrics pycocotools pandas pillow tabulate

## 2. Pipeline smoke test (about 1-2 min, synthetic data)
Proves the code runs on this machine. The numbers are meaningless.

In [ ]:
!python ml/scripts/smoke_test.py

## 3. Download + extract RDD2022 (large; takes a while)

In [ ]:
!python scripts/download_rdd2022.py
!find data/raw/RDD2022 -maxdepth 3 -type d | head -40   # SEND this

## 4. Inventory, audit, split

In [ ]:
!python scripts/build_dataset_inventory.py   # SEND
!python scripts/audit_report.py
!cat docs/DATA_AUDIT.md                          # SEND
!python -m ml.src.data.split_manifest --seed 42

## 5. Look at the ground truth (boxes must sit on the damage)

In [ ]:
!python ml/scripts/visualize_annotations.py --n 8
from PIL import Image; import glob
for p in sorted(glob.glob('docs/figures/gt_samples/*.jpg'))[:6]:
    display(Image.open(p).resize((480, 360)))

## 6. Small training run first (500 images, 5 epochs)

In [ ]:
!python -m ml.src.training.train --config ml/configs/baseline.json   # SEND the per-epoch log

## 7. Full training (only if step 6 shows loss decreasing and mAP > 0)

In [ ]:
!python -m ml.src.training.train --config ml/configs/full.json

## 8. Evaluate on validation + error analysis

In [ ]:
!python -m ml.src.evaluation.evaluate --split val --model-version baseline-001
!python -m ml.src.evaluation.error_analysis --split val --max-images 300
!cat artifacts/evaluation/metrics.json artifacts/evaluation/error_analysis.json   # SEND

## 9. Final test-set number: run ONCE, only after you are done tuning

In [ ]:
# !python -m ml.src.evaluation.evaluate --split test --out artifacts/evaluation/metrics_test.json

## 10. Save results to Drive

In [ ]:
!mkdir -p /content/drive/MyDrive/rdd_results
!cp -r artifacts/checkpoints/best.pt artifacts/checkpoints/training_history.json artifacts/evaluation docs/DATA_AUDIT.md data/splits/manifest.csv /content/drive/MyDrive/rdd_results/ 2>/dev/null
!ls -la /content/drive/MyDrive/rdd_results